# 7日間売上予測MLモデル構築

## 概要
本ノートブックでは、FOODEX_DEMO.BUYER_AGENTスキーマのPOSデータを使用して、
商品カテゴリ（CATEGORY_MEDIUM）別の7日間売上予測モデルを構築します。
ML関連オブジェクトはすべて **FOODEX_DEMO.SALES_ML** スキーマに集約します。

### 実装ステップ
1. **事前準備**: コンピュートプール（コンテナランタイム）の作成
2. **データ取得・前処理**: 日別カテゴリ別売上集計
3. **特徴量エンジニアリング**: ラグ特徴量、曜日等を作成しFeature Storeに登録
4. **モデル学習**: XGBoostで単一モデル構築（カテゴリを特徴量として投入）
5. **評価**: RMSE/MAPEでカテゴリ別精度を可視化
6. **モデル登録**: Snowflake Model Registryに登録（SQL推論可能）
7. **実験トラッキング**: メトリクス・パラメータを記録
8. **ML Observability**: Model Monitorでドリフト監視設定

## Step 0: 事前準備 - コンピュートプール作成

**Snowflake Compute Pool**は、Snowpark Container Services (SPCS) 上でコンテナワークロードを実行するためのコンピュートリソースです。
ML推論やモデルサービングに必要なコンテナランタイムを提供します。

- `instance_family`: CPUまたはGPUインスタンスタイプを指定
- `min_nodes` / `max_nodes`: オートスケール範囲
- `auto_suspend_secs`: アイドル時の自動停止時間

In [ ]:
%%sql -r compute_pool_result
CREATE COMPUTE POOL IF NOT EXISTS SALES_FORECAST_POOL
  MIN_NODES = 1
  MAX_NODES = 3
  INSTANCE_FAMILY = CPU_X64_S
  AUTO_SUSPEND_SECS = 3600
  AUTO_RESUME = TRUE;

In [ ]:
%%sql -r pool_desc
DESCRIBE COMPUTE POOL SALES_FORECAST_POOL;

## Step 1: 環境セットアップとライブラリインポート

**Snowpark ML**はSnowflake上でML開発を行うためのライブラリです。
主要コンポーネント:
- `snowflake.ml.modeling`: scikit-learn互換のMLアルゴリズム
- `snowflake.ml.registry`: モデルレジストリ操作
- `snowflake.ml.feature_store`: 特徴量ストア管理
- `snowflake.ml.tracking`: 実験トラッキング（MLflow互換）

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.snowpark.types import IntegerType, FloatType, StringType
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

session = get_active_session()

In [ ]:
session.sql("USE DATABASE FOODEX_DEMO").collect()
session.sql("USE WAREHOUSE COMPUTE_WH").collect()

session.sql("""
CREATE SCHEMA IF NOT EXISTS FOODEX_DEMO.SALES_ML
    COMMENT = 'ML関連オブジェクト集約スキーマ（売上予測モデル、Feature Store、Model Monitor等）'
""").collect()
session.sql("USE SCHEMA SALES_ML").collect()

## Step 2: データ取得・前処理（Dynamic Table）

**Dynamic Table**を使用して日別×カテゴリ別の売上集計を増分処理で実行します。

### Dynamic Tableのメリット:
- **自動増分処理**: ソーステーブルの変更を検知し、差分のみを処理
- **TARGET_LAG**: データの鮮度を指定（1時間以内に最新化）
- **コスト効率**: フルリフレッシュ不要で計算コストを削減
- **宣言的定義**: SQLで定義するだけで自動管理

In [ ]:
session.sql("""
CREATE OR REPLACE DYNAMIC TABLE FOODEX_DEMO.SALES_ML.DAILY_SALES_BY_CATEGORY
    TARGET_LAG = '10 min'
    WAREHOUSE = COMPUTE_WH
    AS
    SELECT 
        t.TRANSACTION_DATE AS SALES_DATE,
        p.CATEGORY_MEDIUM,
        SUM(t.SALES_AMOUNT) AS DAILY_SALES,
        COUNT(DISTINCT t.TRANSACTION_ID) AS TXN_COUNT,
        SUM(t.QUANTITY) AS TOTAL_QTY
    FROM FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS t
    JOIN FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER p 
        ON t.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.CATEGORY_MEDIUM IS NOT NULL
    GROUP BY t.TRANSACTION_DATE, p.CATEGORY_MEDIUM
""").collect()

print("Dynamic Table created: FOODEX_DEMO.SALES_ML.DAILY_SALES_BY_CATEGORY")
print("- TARGET_LAG: 10 min (ソースデータ変更から10分以内に自動更新)")
print("- 増分処理: Snowflakeが自動で差分のみ処理")

In [ ]:
daily_sales_df = session.table("FOODEX_DEMO.SALES_ML.DAILY_SALES_BY_CATEGORY")

print(f"総レコード数: {daily_sales_df.count()}")
print(f"期間: {daily_sales_df.select(F.min('SALES_DATE'), F.max('SALES_DATE')).collect()}")
print(f"カテゴリ数: {daily_sales_df.select(F.count_distinct('CATEGORY_MEDIUM')).collect()[0][0]}")

daily_sales_df.show(10)

## Step 3: 特徴量エンジニアリング & Feature Store登録

**Snowflake Feature Store**は、MLモデル用の特徴量を一元管理するリポジトリです。

### 主要概念:
- **Entity**: 特徴量の主キー（例: カテゴリ×日付）
- **Feature View**: 特徴量の定義とソースデータのマッピング
- **Feature**: 個々の特徴量カラム

### 作成する特徴量:
- **ラグ特徴量**: 過去7日間の売上（LAG_1〜LAG_7）
- **移動平均**: 7日移動平均
- **曜日フラグ**: 曜日（0-6）、週末フラグ
- **月・四半期**: 季節性捕捉用

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database="FOODEX_DEMO",
    name="SALES_ML",
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST
)

In [ ]:
try:
    category_date_entity = Entity(
        name="CATEGORY_DATE_ENTITY",
        join_keys=["CATEGORY_MEDIUM", "SALES_DATE"],
        desc="Category and date composite entity for sales forecasting"
    )
    fs.register_entity(category_date_entity)
    print("Entity 'CATEGORY_DATE_ENTITY' registered successfully.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Entity already exists, continuing...")
    else:
        raise e

In [ ]:
feature_query = """
WITH daily_agg AS (
    SELECT 
        t.TRANSACTION_DATE AS SALES_DATE,
        p.CATEGORY_MEDIUM,
        SUM(t.SALES_AMOUNT) AS DAILY_SALES,
        COUNT(DISTINCT t.TRANSACTION_ID) AS TXN_COUNT,
        SUM(t.QUANTITY) AS TOTAL_QTY
    FROM FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS t
    JOIN FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER p 
        ON t.PRODUCT_ID = p.PRODUCT_ID
    WHERE p.CATEGORY_MEDIUM IS NOT NULL
    GROUP BY t.TRANSACTION_DATE, p.CATEGORY_MEDIUM
),
lag_features AS (
    SELECT 
        SALES_DATE,
        CATEGORY_MEDIUM,
        DAILY_SALES,
        TXN_COUNT,
        TOTAL_QTY,
        LAG(DAILY_SALES, 1) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_1,
        LAG(DAILY_SALES, 2) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_2,
        LAG(DAILY_SALES, 3) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_3,
        LAG(DAILY_SALES, 7) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_7,
        LAG(DAILY_SALES, 14) OVER (PARTITION BY CATEGORY_MEDIUM ORDER BY SALES_DATE) AS LAG_14,
        AVG(DAILY_SALES) OVER (
            PARTITION BY CATEGORY_MEDIUM 
            ORDER BY SALES_DATE 
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS MA_7,
        DAYOFWEEK(SALES_DATE) AS DAY_OF_WEEK,
        CASE WHEN DAYOFWEEK(SALES_DATE) IN (0, 6) THEN 1 ELSE 0 END AS IS_WEEKEND,
        MONTH(SALES_DATE) AS MONTH_NUM,
        QUARTER(SALES_DATE) AS QUARTER_NUM
    FROM daily_agg
)
SELECT * FROM lag_features
WHERE LAG_14 IS NOT NULL
"""

feature_df = session.sql(feature_query)
feature_df.show(5)

In [ ]:
print("Feature Storeから特徴量テーブルを保存...")
feature_df.write.mode("overwrite").save_as_table("FOODEX_DEMO.SALES_ML.SALES_FORECAST_FEATURES")
print("Feature table saved: FOODEX_DEMO.SALES_ML.SALES_FORECAST_FEATURES")

In [ ]:
from snowflake.ml.feature_store import FeatureView

feature_df = session.sql(feature_query)

try:
    sales_fv = FeatureView(
        name="SALES_FORECAST_FV",
        entities=[category_date_entity],
        feature_df=feature_df,
        refresh_freq="1 day",
        desc="Sales forecast features with lag, moving average, and calendar features"
    )
    
    sales_fv = fs.register_feature_view(
        feature_view=sales_fv,
        version="v2",
        block=True
    )
    print("Feature View 'SALES_FORECAST_FV' (v2) registered successfully.")
    print("- refresh_freq: 1 day (毎日自動更新)")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Feature View already exists, retrieving...")
        sales_fv = fs.get_feature_view("SALES_FORECAST_FV", "v2")
    else:
        raise e

In [ ]:
print("\nRegistered Feature Views:")
for fv in fs.list_feature_views().to_pandas().itertuples():
    print(f"  - {fv.NAME} (version: {fv.VERSION})")

## Step 4: Train/Test 分割（Feature Store経由）

**Feature Storeから特徴量を取得**してTrain/Test分割を行います。

`feature_view.read()` でFeature Viewの最新データを取得できます。
これにより、将来的に複数モデルで同じ特徴量を再利用可能になります。

**重要**: 時系列データでは、ランダム分割ではなく時間ベースの分割を使用します。

In [ ]:
print("Feature Storeから特徴量を取得...")
features_df = fs.read_feature_view(sales_fv)

max_date_result = features_df.select(F.max('SALES_DATE')).collect()
max_date = max_date_result[0][0]
print(f"データの最終日: {max_date}")

test_start_date = max_date - timedelta(days=29)
print(f"テスト開始日: {test_start_date}")

In [ ]:
train_df = features_df.filter(F.col('SALES_DATE') < test_start_date)
test_df = features_df.filter(F.col('SALES_DATE') >= test_start_date)

print(f"Training records: {train_df.count()}")
print(f"Test records: {test_df.count()}")
print("\n※ fs.read_feature_view() でFeature Storeから特徴量を取得")

In [ ]:
feature_cols = ['LAG_1', 'LAG_2', 'LAG_3', 'LAG_7', 'LAG_14', 'MA_7', 
                'DAY_OF_WEEK', 'IS_WEEKEND', 'MONTH_NUM', 'QUARTER_NUM',
                'TXN_COUNT', 'TOTAL_QTY']
target_col = 'DAILY_SALES'
category_col = 'CATEGORY_MEDIUM'

print(f"Feature columns: {feature_cols}")
print(f"Target column: {target_col}")

## Step 5: カテゴリのエンコーディング

**アプローチ**: 単一モデルでカテゴリを特徴量として投入するため、
カテゴリ変数（CATEGORY_MEDIUM）をLabel Encodingで数値化します。

Snowflake MLの`LabelEncoder`を使用することで、
変換ロジックがモデルと一緒に保存され、推論時も同じエンコーディングが適用されます。

In [ ]:
from snowflake.ml.modeling.preprocessing import LabelEncoder

label_encoder = LabelEncoder(
    input_cols=[category_col],
    output_cols=["CATEGORY_ENCODED"]
)

label_encoder.fit(train_df)

train_encoded = label_encoder.transform(train_df)
test_encoded = label_encoder.transform(test_df)

train_encoded.select(category_col, "CATEGORY_ENCODED").distinct().show()

In [ ]:
all_feature_cols = feature_cols + ["CATEGORY_ENCODED"]
print(f"All features (including encoded category): {all_feature_cols}")

## Step 6: 実験トラッキング設定 & モデル学習

**Snowflake Experiment Tracking**はMLflowベースの実験管理機能です。

### 主要機能:
- **Experiment**: 関連するモデル実験をグループ化
- **Run**: 1回の学習実行の記録（パラメータ、メトリクス、モデル）
- **Autolog**: 主要MLフレームワークの自動ログ機能

### XGBoostの選択理由:
- 時系列の非線形パターンを捕捉可能
- カテゴリ特徴量を効率的に処理
- 解釈性の高い特徴量重要度を提供

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.modeling.xgboost import XGBRegressor
import time

exp = ExperimentTracking(session=session)
exp.set_experiment("SALES_FORECAST_EXPERIMENT")

print(f"Experiment set: SALES_FORECAST_EXPERIMENT")

In [ ]:
model_params = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.15,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42
}

try:
    exp.end_run()
except:
    pass

exp.start_run(f"xgboost_run_{int(time.time())}")
exp.log_params(model_params)
exp.log_param("feature_count", len(all_feature_cols))
exp.log_param("target_column", target_col)
print("Experiment run started and parameters logged.")

In [ ]:
xgb_model = XGBRegressor(
    input_cols=all_feature_cols,
    label_cols=[target_col],
    output_cols=["PREDICTED_SALES"],
    **model_params
)

print("Training XGBoost model...")
start_time = time.time()
xgb_model.fit(train_encoded)
training_time = time.time() - start_time
print(f"Training completed in {training_time:.2f} seconds")

exp.log_metric("training_time_sec", training_time)

## Step 7: 評価指標計算（RMSE, MAPE）& カテゴリ別可視化

### 評価指標:
- **RMSE (Root Mean Squared Error)**: 予測誤差の大きさを評価。単位は目的変数と同じ（円）
- **MAPE (Mean Absolute Percentage Error)**: 予測誤差を%で評価。スケールに依存しない比較が可能

### 可視化:
- カテゴリ別の予測精度を棒グラフで表示
- 実績 vs 予測の散布図

In [ ]:
predictions_df = xgb_model.predict(test_encoded.select(all_feature_cols + [target_col, 'SALES_DATE', 'CATEGORY_MEDIUM']))

pred_pd = predictions_df.select(
    'SALES_DATE', 'CATEGORY_MEDIUM', 'DAILY_SALES', 'PREDICTED_SALES'
).to_pandas()

pred_pd['ERROR'] = pred_pd['DAILY_SALES'] - pred_pd['PREDICTED_SALES']
pred_pd['ABS_PCT_ERROR'] = np.abs(pred_pd['ERROR']) / pred_pd['DAILY_SALES'] * 100

pred_pd.head(10)

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

overall_rmse = np.sqrt(mean_squared_error(pred_pd['DAILY_SALES'], pred_pd['PREDICTED_SALES']))
overall_mape = mean_absolute_percentage_error(pred_pd['DAILY_SALES'], pred_pd['PREDICTED_SALES']) * 100

print(f"=== Overall Metrics ===")
print(f"RMSE: {overall_rmse:,.2f} 円")
print(f"MAPE: {overall_mape:.2f} %")

exp.log_metric("overall_rmse", overall_rmse)
exp.log_metric("overall_mape", overall_mape)

In [ ]:
category_metrics = pred_pd.groupby('CATEGORY_MEDIUM').apply(
    lambda x: pd.Series({
        'RMSE': np.sqrt(mean_squared_error(x['DAILY_SALES'], x['PREDICTED_SALES'])),
        'MAPE': mean_absolute_percentage_error(x['DAILY_SALES'], x['PREDICTED_SALES']) * 100,
        'Sample_Count': len(x)
    })
).reset_index()

category_metrics = category_metrics.sort_values('MAPE')
print("=== Category-wise Metrics ===")
print(category_metrics.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

category_metrics = pred_pd.groupby('CATEGORY_MEDIUM').apply(
    lambda x: pd.Series({
        'RMSE': np.sqrt(mean_squared_error(x['DAILY_SALES'], x['PREDICTED_SALES'])),
        'MAPE': mean_absolute_percentage_error(x['DAILY_SALES'], x['PREDICTED_SALES']) * 100,
        'Sample_Count': len(x)
    })
).reset_index()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

ax1 = axes[0, 0]
category_metrics_sorted = category_metrics.sort_values('RMSE', ascending=True)
ax1.barh(category_metrics_sorted['CATEGORY_MEDIUM'], category_metrics_sorted['RMSE'], color='steelblue')
ax1.set_xlabel('RMSE (円)')
ax1.set_title('カテゴリ別 RMSE')
ax1.grid(axis='x', alpha=0.3)

ax2 = axes[0, 1]
category_metrics_sorted = category_metrics.sort_values('MAPE', ascending=True)
ax2.barh(category_metrics_sorted['CATEGORY_MEDIUM'], category_metrics_sorted['MAPE'], color='coral')
ax2.set_xlabel('MAPE (%)')
ax2.set_title('カテゴリ別 MAPE')
ax2.grid(axis='x', alpha=0.3)

ax3 = axes[1, 0]
ax3.scatter(pred_pd['DAILY_SALES'], pred_pd['PREDICTED_SALES'], alpha=0.3, s=10)
max_val = max(pred_pd['DAILY_SALES'].max(), pred_pd['PREDICTED_SALES'].max())
ax3.plot([0, max_val], [0, max_val], 'r--', label='Perfect Prediction')
ax3.set_xlabel('Actual Sales (円)')
ax3.set_ylabel('Predicted Sales (円)')
ax3.set_title('Actual vs Predicted')
ax3.legend()
ax3.grid(alpha=0.3)

ax4 = axes[1, 1]
top_categories = category_metrics.head(5)['CATEGORY_MEDIUM'].tolist()
for cat in top_categories:
    cat_data = pred_pd[pred_pd['CATEGORY_MEDIUM'] == cat].sort_values('SALES_DATE')
    ax4.plot(cat_data['SALES_DATE'], cat_data['DAILY_SALES'], label=f'{cat} (実績)', linestyle='-')
    ax4.plot(cat_data['SALES_DATE'], cat_data['PREDICTED_SALES'], label=f'{cat} (予測)', linestyle='--', alpha=0.7)
ax4.set_xlabel('Date')
ax4.set_ylabel('Sales (円)')
ax4.set_title('Top 5 Categories: Actual vs Predicted')
ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax4.grid(alpha=0.3)
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
run.end_run()
print("Experiment run ended successfully.")

## Step 8: Snowflake Model Registryへの登録

**Snowflake Model Registry**は、MLモデルのバージョン管理と展開を行う中央リポジトリです。

### 主要パラメータ:
- **model_name**: モデルの識別名
- **version_name**: バージョン管理用の名前
- **target_platforms**: 推論先プラットフォーム
  - `TargetPlatform.WAREHOUSE`: SQLウェアハウスで推論（SQL関数として呼び出し可能）
  - `TargetPlatform.SNOWPARK_CONTAINER_SERVICES`: SPCSで推論
- **sample_input_data**: スキーマ推論用のサンプルデータ
- **metrics**: モデルのパフォーマンス指標

### SQL推論のメリット:
- SQLクエリから直接予測を取得
- 既存のBIツールやレポートと容易に統合
- Snowflakeのスケーラビリティを活用

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.registry.model_version import TargetPlatform

registry = Registry(
    session=session,
    database_name="FOODEX_DEMO",
    schema_name="SALES_ML"
)

print("Model Registry initialized.")

In [ ]:
sample_input = train_encoded.select(all_feature_cols).limit(100)

model_version = registry.log_model(
    model=xgb_model,
    model_name="SALES_FORECAST_MODEL",
    version_name="v1",
    target_platforms=[TargetPlatform.WAREHOUSE],
    sample_input_data=sample_input,
    metrics={
        "overall_rmse": overall_rmse,
        "overall_mape": overall_mape,
        "training_time_sec": training_time
    },
    comment="7-day sales forecast model by category using XGBoost"
)

print(f"Model registered: FOODEX_DEMO.SALES_ML.{model_version.model_name} (version: {model_version.version_name})")

In [ ]:
print("\nRegistered Models:")
for model in registry.models():
    print(f"  - {model.name}")
    for version in model.versions():
        print(f"      Version: {version.version_name}")
        print(f"      Metrics: {version.get_metric('overall_rmse')}, {version.get_metric('overall_mape')}")

### SQL推論のテスト

Model Registryに登録したモデルは、SQL関数として呼び出せます。
`MODEL_NAME!PREDICT()`の形式でSQLから推論を実行できます。

In [ ]:
test_with_encoding = label_encoder.transform(
    session.table("FOODEX_DEMO.SALES_ML.SALES_FORECAST_FEATURES").filter(F.col("SALES_DATE") >= test_start_date)
)

sql_test_result = model_version.run(test_with_encoding.select(all_feature_cols).limit(20))
print("Model Registry SQL Inference Test:")
sql_test_result.show()

## Step 9: ML Observability - Model Monitor設定

**Snowflake Model Monitor**は、本番環境でのモデルパフォーマンスとデータドリフトを継続監視するML Observability機能です。

### 主要機能:
- **予測統計**: 予測分布、カウント、NULL率の追跡
- **ドリフト検出**: PSI (Population Stability Index) でデータドリフトを検出
- **パフォーマンス指標**: RMSE、MAPE等の推論精度を継続監視
- **セグメント別分析**: カテゴリ別にメトリクスを分割表示

### 前提条件:
- Model Registryに登録済みのモデル
- 推論ログテーブル（timestamp, 特徴量, 予測値, 実績値）
- TIMESTAMP_NTZ型のタイムスタンプカラム

In [ ]:
inference_log_df = session.sql(f"""
    SELECT 
        sf.*,
        PREDICTED_SALES,
        CURRENT_TIMESTAMP()::TIMESTAMP_NTZ AS INFERENCE_TIMESTAMP
    FROM SALES_FORECAST_FEATURES sf
    INNER JOIN (
        SELECT SALES_DATE, CATEGORY_MEDIUM, PREDICTED_SALES
        FROM (
            SELECT 
                SALES_DATE,
                CATEGORY_MEDIUM,
                DAILY_SALES,
                LAG_1, LAG_2, LAG_3, LAG_7, LAG_14, MA_7,
                DAY_OF_WEEK, IS_WEEKEND, MONTH_NUM, QUARTER_NUM,
                TXN_COUNT, TOTAL_QTY,
                {predictions_df.select('PREDICTED_SALES').limit(1).collect()[0][0]} AS PREDICTED_SALES
            FROM SALES_FORECAST_FEATURES
            WHERE SALES_DATE >= '{test_start_date}'
        )
    ) pred
    ON sf.SALES_DATE = pred.SALES_DATE AND sf.CATEGORY_MEDIUM = pred.CATEGORY_MEDIUM
""")
print(inference_log_df.count())

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE FOODEX_DEMO.SALES_ML.SALES_FORECAST_INFERENCE_LOG AS
SELECT 
    sf.SALES_DATE,
    sf.CATEGORY_MEDIUM,
    sf.DAILY_SALES,
    sf.TXN_COUNT,
    sf.TOTAL_QTY,
    sf.LAG_1,
    sf.LAG_2,
    sf.LAG_3,
    sf.LAG_7,
    sf.LAG_14,
    sf.MA_7,
    sf.DAY_OF_WEEK,
    sf.IS_WEEKEND,
    sf.MONTH_NUM,
    sf.QUARTER_NUM,
    pred.PREDICTED_SALES,
    sf.DAILY_SALES AS ACTUAL_SALES,
    DATEADD('day', ROW_NUMBER() OVER (ORDER BY sf.SALES_DATE, sf.CATEGORY_MEDIUM), '2026-03-01')::TIMESTAMP_NTZ AS INFERENCE_TIMESTAMP
FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_FEATURES sf
INNER JOIN (
    SELECT SALES_DATE, CATEGORY_MEDIUM, PREDICTED_SALES 
    FROM ({predictions_df.select('SALES_DATE', 'CATEGORY_MEDIUM', 'PREDICTED_SALES').queries['queries'][0]})
) pred
ON sf.SALES_DATE = pred.SALES_DATE AND sf.CATEGORY_MEDIUM = pred.CATEGORY_MEDIUM
""").collect()

print("Inference log table created: FOODEX_DEMO.SALES_ML.SALES_FORECAST_INFERENCE_LOG")

In [ ]:
session.sql("SELECT * FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_INFERENCE_LOG LIMIT 5").show()

### Baseline テーブルの作成

ドリフト検出のために、学習データの分布をBaseline（基準）として保存します。
推論データの分布がBaselineから乖離した場合、ドリフトとして検出されます。

In [ ]:
session.sql(f"""
CREATE OR REPLACE TABLE FOODEX_DEMO.SALES_ML.SALES_FORECAST_BASELINE AS
SELECT 
    SALES_DATE,
    CATEGORY_MEDIUM,
    DAILY_SALES,
    TXN_COUNT,
    TOTAL_QTY,
    LAG_1,
    LAG_2,
    LAG_3,
    LAG_7,
    LAG_14,
    MA_7,
    DAY_OF_WEEK,
    IS_WEEKEND,
    MONTH_NUM,
    QUARTER_NUM,
    0.0 AS PREDICTED_SALES,
    DAILY_SALES AS ACTUAL_SALES,
    SALES_DATE::TIMESTAMP_NTZ AS INFERENCE_TIMESTAMP
FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_FEATURES
WHERE SALES_DATE < '{test_start_date}'
""").collect()

print("Baseline table created: FOODEX_DEMO.SALES_ML.SALES_FORECAST_BASELINE")

### Model Monitor の作成

以下のパラメータでモニターを設定します:
- **MODEL**: 監視対象モデル
- **SOURCE**: 推論ログテーブル  
- **BASELINE**: ドリフト検出用の基準データ
- **SEGMENT_COLUMNS**: カテゴリ別にメトリクスを分割
- **PREDICTION_SCORE_COLUMNS**: 予測値カラム
- **ACTUAL_SCORE_COLUMNS**: 実績値カラム（パフォーマンス計算用）

In [ ]:
%%sql -r monitor_result
CREATE OR REPLACE MODEL MONITOR SALES_FORECAST_MONITOR WITH
    MODEL = FOODEX_DEMO.SALES_ML.SALES_FORECAST_MODEL
    VERSION = 'v1'
    FUNCTION = 'PREDICT'
    SOURCE = FOODEX_DEMO.SALES_ML.SALES_FORECAST_INFERENCE_LOG
    WAREHOUSE = COMPUTE_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '1 day'
    TIMESTAMP_COLUMN = INFERENCE_TIMESTAMP
    PREDICTION_SCORE_COLUMNS = ('PREDICTED_SALES')
    ACTUAL_SCORE_COLUMNS = ('ACTUAL_SALES')
    BASELINE = FOODEX_DEMO.SALES_ML.SALES_FORECAST_BASELINE
    SEGMENT_COLUMNS = ('CATEGORY_MEDIUM');

In [ ]:
%%sql -r monitor_desc
DESC MODEL MONITOR SALES_FORECAST_MONITOR;

### Model Monitor メトリクスのクエリ

**利用可能なメトリクス:**
- **ドリフト**: PSI (Population Stability Index), KL_DIVERGENCE
- **パフォーマンス（回帰）**: RMSE, MAE, MAPE
- **統計**: COUNT, NULL_COUNT, MEAN, STDDEV

In [ ]:
%%sql -r rmse_metrics
SELECT * 
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    'SALES_FORECAST_MONITOR',
    'RMSE',
    'DAY',
    '2026-03-01'::TIMESTAMP_NTZ,
    '2026-03-11'::TIMESTAMP_NTZ
))
ORDER BY TIMESTAMP_VALUE;

In [ ]:
%%sql -r mape_metrics
SELECT * 
FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
    'SALES_FORECAST_MONITOR',
    'MAPE',
    'DAY',
    '2026-03-01'::TIMESTAMP_NTZ,
    '2026-03-11'::TIMESTAMP_NTZ
))
ORDER BY TIMESTAMP_VALUE;

### カテゴリ別のドリフト検出

PSI (Population Stability Index) を使って特徴量のドリフトを検出します。
一般的な閾値:
- PSI < 0.1: 変化なし
- 0.1 ≤ PSI < 0.25: 軽度のドリフト（監視継続）
- PSI ≥ 0.25: 重大なドリフト（要対応）

In [ ]:
%%sql -r drift_lag1
SELECT * 
FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
    'SALES_FORECAST_MONITOR',
    'PSI',
    'LAG_1',
    'DAY',
    '2026-03-01'::TIMESTAMP_NTZ,
    '2026-03-11'::TIMESTAMP_NTZ
))
ORDER BY TIMESTAMP_VALUE;

In [ ]:
categories = session.sql("""
    SELECT DISTINCT CATEGORY_MEDIUM FROM FOODEX_DEMO.SALES_ML.SALES_FORECAST_INFERENCE_LOG ORDER BY 1 LIMIT 5
""").to_pandas()['CATEGORY_MEDIUM'].tolist()

print("Querying category-specific metrics...")
for cat in categories[:3]:
    print(f"\n=== {cat} ===")
    segment_filter = f'{{"SEGMENTS": [{{"column": "CATEGORY_MEDIUM", "value": "{cat}"}}]}}'
    try:
        result = session.sql(f"""
            SELECT TIMESTAMP_VALUE, METRIC_VALUE 
            FROM TABLE(MODEL_MONITOR_STAT_METRIC(
                'SALES_FORECAST_MONITOR',
                'COUNT',
                'DAY',
                '2026-03-01'::TIMESTAMP_NTZ,
                '2026-03-11'::TIMESTAMP_NTZ,
                '{segment_filter}'
            ))
            ORDER BY TIMESTAMP_VALUE
            LIMIT 5
        """).to_pandas()
        print(result.to_string(index=False))
    except Exception as e:
        print(f"Note: Segment query may take time to populate: {str(e)[:50]}...")

### Model Monitor 可視化

カテゴリ別のモニタリングメトリクスを可視化します。
Snowsight UI でより詳細なダッシュボードも確認できます:
`AI&ML → Models → SALES_FORECAST_MODEL → Monitors`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

try:
    rmse_data = session.sql("""
        SELECT TIMESTAMP_VALUE, METRIC_VALUE as RMSE
        FROM TABLE(MODEL_MONITOR_PERFORMANCE_METRIC(
            'SALES_FORECAST_MONITOR', 'RMSE', 'DAY',
            '2026-03-01'::TIMESTAMP_NTZ, '2026-03-11'::TIMESTAMP_NTZ
        )) ORDER BY TIMESTAMP_VALUE
    """).to_pandas()
    
    if len(rmse_data) > 0:
        axes[0].plot(rmse_data['TIMESTAMP_VALUE'], rmse_data['RMSE'], marker='o', color='steelblue')
        axes[0].set_xlabel('Date')
        axes[0].set_ylabel('RMSE')
        axes[0].set_title('Model Monitor: RMSE Over Time')
        axes[0].grid(alpha=0.3)
        axes[0].tick_params(axis='x', rotation=45)
    else:
        axes[0].text(0.5, 0.5, 'Data populating...\nRefresh in 1 hour', ha='center', va='center', fontsize=12)
        axes[0].set_title('Model Monitor: RMSE Over Time')
except Exception as e:
    axes[0].text(0.5, 0.5, f'Monitor initializing...\n{str(e)[:30]}', ha='center', va='center', fontsize=10)
    axes[0].set_title('Model Monitor: RMSE Over Time')

try:
    drift_data = session.sql("""
        SELECT TIMESTAMP_VALUE, METRIC_VALUE as PSI
        FROM TABLE(MODEL_MONITOR_DRIFT_METRIC(
            'SALES_FORECAST_MONITOR', 'PSI', 'LAG_1', 'DAY',
            '2026-03-01'::TIMESTAMP_NTZ, '2026-03-11'::TIMESTAMP_NTZ
        )) ORDER BY TIMESTAMP_VALUE
    """).to_pandas()
    
    if len(drift_data) > 0:
        axes[1].bar(drift_data['TIMESTAMP_VALUE'].astype(str), drift_data['PSI'], color='coral')
        axes[1].axhline(y=0.1, color='orange', linestyle='--', label='Warning (0.1)')
        axes[1].axhline(y=0.25, color='red', linestyle='--', label='Critical (0.25)')
        axes[1].set_xlabel('Date')
        axes[1].set_ylabel('PSI')
        axes[1].set_title('Model Monitor: Feature Drift (LAG_1)')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        axes[1].tick_params(axis='x', rotation=45)
    else:
        axes[1].text(0.5, 0.5, 'Baseline comparison processing...', ha='center', va='center', fontsize=12)
        axes[1].set_title('Model Monitor: Feature Drift (LAG_1)')
except Exception as e:
    axes[1].text(0.5, 0.5, f'Drift metrics initializing...\n{str(e)[:30]}', ha='center', va='center', fontsize=10)
    axes[1].set_title('Model Monitor: Feature Drift (LAG_1)')

plt.tight_layout()
plt.show()

## まとめ

本ノートブックでは、Snowflake MLプラットフォームを使用して、商品カテゴリ別の7日間売上予測モデルを構築しました。

### 実装した機能:

| ステップ | Snowflake ML機能 | 説明 |
|---------|-----------------|------|
| 0 | Compute Pool | SPCSコンテナランタイムの準備 |
| 1-2 | Snowpark + Dynamic Table | セッション管理・増分データ処理 |
| 3 | **Feature Store** | 特徴量の一元管理（Entity, FeatureView, 日次自動更新） |
| 4 | **Feature Store → read()** | Feature Storeから特徴量取得してTrain/Test分割 |
| 5 | snowflake.ml.modeling | LabelEncoder, XGBRegressorでモデル構築 |
| 6 | Experiment Tracking | パラメータ・メトリクスの記録 |
| 7 | Model Registry | モデル登録、SQL推論（`target_platforms=WAREHOUSE`） |
| 8 | Model Monitor | ドリフト検出・パフォーマンス監視 |

### アーキテクチャ:

```
POSデータ → Dynamic Table → Feature Store (日次更新)
                                   ↓
                            Task (毎日実行)
                                   ↓
                        Feature Store.read()
                                   ↓
                          モデルトレーニング
                                   ↓
                          Model Registry
```

### Feature Storeのメリット:
- **複数モデルで特徴量を共有可能**
- **特徴量のバージョン管理・監査**
- **日次自動更新（refresh_freq='1 day'）**

In [ ]:
print("=" * 60)
print("       7日間売上予測モデル構築完了")
print("=" * 60)
print(f"\n[評価結果]")
print(f"  Overall RMSE: {overall_rmse:,.2f} 円")
print(f"  Overall MAPE: {overall_mape:.2f} %")
print(f"\n[作成オブジェクト - FOODEX_DEMO.SALES_ML スキーマ]")
print(f"  Model: FOODEX_DEMO.SALES_ML.SALES_FORECAST_MODEL (v1)")
print(f"  Monitor: FOODEX_DEMO.SALES_ML.SALES_FORECAST_MONITOR")
print(f"  Feature View: FOODEX_DEMO.SALES_ML.SALES_FORECAST_FV (v1)")
print(f"  Dynamic Table: FOODEX_DEMO.SALES_ML.DAILY_SALES_BY_CATEGORY")
print(f"  Tables: SALES_FORECAST_FEATURES, SALES_FORECAST_INFERENCE_LOG, SALES_FORECAST_BASELINE")
print(f"\n[SQL推論テスト]")
print(f"  model_version.run() でテスト済み")
print(f"\n[ML Observability]")
print(f"  - パフォーマンス監視: RMSE, MAPE")
print(f"  - ドリフト検出: PSI (LAG特徴量)")
print(f"  - セグメント分析: CATEGORY_MEDIUM別")

### シンプル特徴量セット（増分更新対応）

ウィンドウ関数を使わないシンプルな特徴量セットを作成します。
こちらは**INCREMENTAL**モードで増分更新が可能です。

**含まれる特徴量:**
- 日次売上、トランザクション数、数量
- 曜日、週末フラグ、月、四半期（カレンダー特徴量）

In [ ]:
simple_feature_query = """
SELECT 
    t.TRANSACTION_DATE AS SALES_DATE,
    p.CATEGORY_MEDIUM,
    SUM(t.SALES_AMOUNT) AS DAILY_SALES,
    COUNT(DISTINCT t.TRANSACTION_ID) AS TXN_COUNT,
    SUM(t.QUANTITY) AS TOTAL_QTY,
    DAYOFWEEK(t.TRANSACTION_DATE) AS DAY_OF_WEEK,
    CASE WHEN DAYOFWEEK(t.TRANSACTION_DATE) IN (0, 6) THEN 1 ELSE 0 END AS IS_WEEKEND,
    MONTH(t.TRANSACTION_DATE) AS MONTH_NUM,
    QUARTER(t.TRANSACTION_DATE) AS QUARTER_NUM,
    YEAR(t.TRANSACTION_DATE) AS YEAR_NUM
FROM FOODEX_DEMO.BUYER_AGENT.ID_POS_TRANSACTIONS t
JOIN FOODEX_DEMO.BUYER_AGENT.PRODUCT_MASTER p 
    ON t.PRODUCT_ID = p.PRODUCT_ID
WHERE p.CATEGORY_MEDIUM IS NOT NULL
GROUP BY t.TRANSACTION_DATE, p.CATEGORY_MEDIUM
"""

simple_feature_df = session.sql(simple_feature_query)
simple_feature_df.show(5)

In [ ]:
simple_entity = Entity(
    name="CATEGORY_DATE_SIMPLE",
    join_keys=["CATEGORY_MEDIUM", "SALES_DATE"],
    desc="Simple category-date entity (incremental refresh supported)"
)

try:
    fs.register_entity(simple_entity)
    print("Entity 'CATEGORY_DATE_SIMPLE' registered.")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Entity already exists, continuing...")
    else:
        raise e

In [ ]:
try:
    simple_fv = FeatureView(
        name="SALES_SIMPLE_FV",
        entities=[simple_entity],
        feature_df=simple_feature_df,
        refresh_freq="1 day",
        desc="Simple sales features - incremental refresh supported"
    )
    
    simple_fv = fs.register_feature_view(
        feature_view=simple_fv,
        version="v1",
        block=True
    )
    print("Feature View 'SALES_SIMPLE_FV' registered.")
    print("- refresh_freq: 1 day")
    print("- mode: INCREMENTAL (増分更新対応)")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Feature View already exists, retrieving...")
        simple_fv = fs.get_feature_view("SALES_SIMPLE_FV", "v1")
    else:
        raise e

In [ ]:
print("\n=== 登録済みFeature View一覧 ===")
for fv in fs.list_feature_views().to_pandas().itertuples():
    print(f"  - {fv.NAME} (version: {fv.VERSION})")

print("\n=== Feature View比較 ===")
print("1. SALES_FORECAST_FV: LAG/移動平均あり → FULL refresh")
print("2. SALES_SIMPLE_FV: カレンダー特徴量のみ → INCREMENTAL refresh")